# Policy Gradients - Learning to Act Through Direct Optimization

[![Open In Colab](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/rl-policy-gradients.ipynb)](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/rl-policy-gradients.ipynb)

This notebook introduces **policy gradient methods** - a family of reinforcement learning algorithms that directly optimize the policy (the agent's behavior) using gradient ascent on expected rewards.

## What Are Policy Gradients?

Policy gradient methods take a fundamentally different approach from value-based methods like Q-learning:

- **Value-based** (Q-learning, DQN): Learn Q(s,a) → derive policy by choosing max Q
- **Policy-based** (Policy Gradients): Directly learn π(a|s) with gradient ascent

### Why Policy Gradients?

**Advantages:**
1. **Stochastic policies**: Can learn probabilistic actions (important for games like rock-paper-scissors)
2. **Continuous action spaces**: Works naturally with continuous actions (robot control)
3. **Convergence guarantees**: Gradient ascent converges to local optima
4. **Simpler for some problems**: No need to learn value function explicitly

**Trade-offs:**
1. **High variance**: Policy gradient estimates can be noisy
2. **Sample inefficient**: Often needs many episodes to learn
3. **Local optima**: Can get stuck in suboptimal policies

### What We'll Learn

1. **Policy Gradient Theorem** - The math behind why this works
2. **REINFORCE** - The simplest policy gradient algorithm
3. **Baselines** - How to reduce variance and stabilize training
4. **Actor-Critic** - Combining policy and value functions
5. **PPO Basics** - Modern policy optimization

## Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym
import matplotlib.pyplot as plt
from collections import deque
from typing import List, Tuple

from aiml_notebooks import set_seed, get_device

# Set random seed for reproducibility
set_seed(42)

# Use CPU for RL (simpler, MPS can have issues with some ops)
device = get_device(prefer_cpu=True)
print(f"Using device: {device}")

## 1. The CartPole Environment

We'll use **CartPole-v1** - a classic RL benchmark where you balance a pole on a moving cart.

**Goal**: Keep the pole upright by moving the cart left or right.

**State**: 4 continuous values (cart position, cart velocity, pole angle, pole angular velocity)

**Actions**: 2 discrete actions (move left: 0, move right: 1)

**Reward**: +1 for each timestep the pole stays upright

**Episode ends when**: Pole falls (angle > 15°) or cart moves off screen or 500 steps reached

In [ ]:
# Create environment
env = gym.make('CartPole-v1')

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

print(f"State dimension: {state_dim}")
print(f"Action dimension: {action_dim}")
print(f"Max episode length: 500")
print(f"Success threshold: 475 (average reward over 100 episodes)")

Let's test a random policy to see baseline performance:

In [ ]:
def test_random_policy(env, num_episodes=10):
    """Test a random policy."""
    episode_rewards = []
    
    for _ in range(num_episodes):
        state, _ = env.reset(seed=42)
        episode_reward = 0
        done = False
        
        while not done:
            action = env.action_space.sample()  # Random action
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            episode_reward += reward
        
        episode_rewards.append(episode_reward)
    
    return episode_rewards

random_rewards = test_random_policy(env, num_episodes=100)
print(f"Random policy average reward: {np.mean(random_rewards):.2f} ± {np.std(random_rewards):.2f}")
print(f"This is our baseline to beat!")

## 2. Policy Network

A **policy network** is a neural network that outputs action probabilities given a state.

For discrete actions, we use:
- Input: state (4 dimensions)
- Hidden layers: learn features from state
- Output: logits for each action (2 dimensions)
- **Softmax**: convert logits to probabilities

$$\pi_\theta(a|s) = \text{softmax}(f_\theta(s))$$

where $\theta$ are the network parameters we'll optimize.

In [ ]:
class PolicyNetwork(nn.Module):
    """Simple policy network for discrete actions."""
    
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)
    
    def forward(self, state):
        """Forward pass: state -> action logits."""
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        logits = self.fc3(x)
        return logits
    
    def get_action(self, state):
        """Sample an action from the policy."""
        logits = self.forward(state)
        dist = Categorical(logits=logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return action.item(), log_prob

# Create policy network
policy = PolicyNetwork(state_dim, action_dim).to(device)
print(f"Policy network has {sum(p.numel() for p in policy.parameters())} parameters")

Let's visualize what the untrained policy outputs:

In [ ]:
# Sample a random state
state, _ = env.reset(seed=42)
state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)

# Get action probabilities
with torch.no_grad():
    logits = policy(state_tensor)
    probs = F.softmax(logits, dim=-1)

print(f"State: {state}")
print(f"Action probabilities: {probs.cpu().numpy()[0]}")
print(f"Before training, the policy is nearly random (close to 0.5/0.5)")

## 3. Policy Gradient Theorem

The **policy gradient theorem** tells us how to improve our policy using gradient ascent.

### The Objective

We want to maximize the expected return:

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]$$

where:
- $\tau$ is a trajectory (sequence of states, actions, rewards)
- $R(\tau) = \sum_{t=0}^T r_t$ is the total reward
- $\pi_\theta$ is our policy with parameters $\theta$

### The Gradient

The policy gradient theorem gives us:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot R(\tau)\right]$$

**Intuition**: 
- If a trajectory has **high reward** $R(\tau)$, **increase** the probability of taking those actions
- If a trajectory has **low reward**, **decrease** the probability
- $\nabla_\theta \log \pi_\theta(a_t|s_t)$ is the direction to change $\theta$ to increase $\pi_\theta(a_t|s_t)$

### Monte Carlo Estimation

We estimate the gradient using sampled trajectories:

$$\nabla_\theta J(\theta) \approx \frac{1}{N}\sum_{i=1}^N \sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t^{(i)}|s_t^{(i)}) \cdot R(\tau^{(i)})$$

This is the foundation of **REINFORCE**!

Let's visualize the log probability computation:

In [ ]:
# Sample an action and compute log probability
action, log_prob = policy.get_action(state_tensor)

print(f"Sampled action: {action}")
print(f"Log probability: {log_prob.item():.4f}")
print(f"Probability: {torch.exp(log_prob).item():.4f}")
print(f"\nThe log_prob will be used to compute gradients!")

## 4. REINFORCE Algorithm

**REINFORCE** is the simplest policy gradient algorithm. Here's the complete procedure:

1. Initialize policy network $\pi_\theta$
2. For each episode:
   - Collect trajectory $\tau = (s_0, a_0, r_0, s_1, a_1, r_1, ...)$ by following $\pi_\theta$
   - Compute returns $G_t = \sum_{t'=t}^T r_{t'}$ for each timestep
   - Compute policy gradient: $\nabla_\theta J \approx \sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t$
   - Update: $\theta \leftarrow \theta + \alpha \nabla_\theta J$

**Key insight**: We use returns $G_t$ (future rewards from time $t$) instead of total reward $R(\tau)$ to reduce variance.

In [ ]:
def compute_returns(rewards, gamma=0.99):
    """Compute discounted returns for each timestep.
    
    G_t = r_t + γ*r_{t+1} + γ²*r_{t+2} + ...
    """
    returns = []
    G = 0
    
    # Compute returns backwards (from end to start)
    for reward in reversed(rewards):
        G = reward + gamma * G
        returns.insert(0, G)
    
    return returns

# Example: compute returns for a simple episode
example_rewards = [1, 1, 1, 1, 1]  # 5 timesteps
example_returns = compute_returns(example_rewards, gamma=0.99)

print("Rewards:", example_rewards)
print("Returns:", [f"{g:.2f}" for g in example_returns])
print("\nNotice: earlier rewards have higher returns (they contribute to more future rewards)")

Now let's implement the full REINFORCE algorithm:

In [ ]:
class REINFORCE:
    """REINFORCE algorithm (Monte Carlo Policy Gradient)."""
    
    def __init__(self, policy, lr=0.001, gamma=0.99):
        self.policy = policy
        self.optimizer = optim.Adam(policy.parameters(), lr=lr)
        self.gamma = gamma
        
        # Storage for episode
        self.log_probs = []
        self.rewards = []
    
    def select_action(self, state):
        """Select action and store log probability."""
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        action, log_prob = self.policy.get_action(state)
        
        self.log_probs.append(log_prob)
        return action
    
    def store_reward(self, reward):
        """Store reward for current timestep."""
        self.rewards.append(reward)
    
    def update(self):
        """Update policy using collected episode."""
        # Compute returns
        returns = compute_returns(self.rewards, self.gamma)
        returns = torch.FloatTensor(returns).to(device)
        
        # Normalize returns (helps with stability)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        
        # Compute policy gradient loss
        policy_loss = []
        for log_prob, G in zip(self.log_probs, returns):
            policy_loss.append(-log_prob * G)  # Negative for gradient ascent
        
        policy_loss = torch.stack(policy_loss).sum()
        
        # Update policy
        self.optimizer.zero_grad()
        policy_loss.backward()
        self.optimizer.step()
        
        # Clear episode storage
        self.log_probs = []
        self.rewards = []
        
        return policy_loss.item()

# Create REINFORCE agent
policy_net = PolicyNetwork(state_dim, action_dim).to(device)
agent = REINFORCE(policy_net, lr=0.001, gamma=0.99)
print("REINFORCE agent created!")

## 5. Training REINFORCE

Now let's train our REINFORCE agent on CartPole!

In [ ]:
def train_reinforce(agent, env, num_episodes=500, print_every=50):
    """Train REINFORCE agent."""
    episode_rewards = []
    running_reward = deque(maxlen=100)
    
    for episode in range(num_episodes):
        state, _ = env.reset()
        episode_reward = 0
        done = False
        
        # Collect episode
        while not done:
            action = agent.select_action(state)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            agent.store_reward(reward)
            episode_reward += reward
        
        # Update policy
        agent.update()
        
        # Track progress
        episode_rewards.append(episode_reward)
        running_reward.append(episode_reward)
        
        if (episode + 1) % print_every == 0:
            avg_reward = np.mean(running_reward)
            print(f"Episode {episode+1}/{num_episodes} | Avg Reward: {avg_reward:.2f}")
            
            # Check if solved (average reward > 475 over last 100 episodes)
            if avg_reward > 475:
                print(f"\n✓ Solved in {episode+1} episodes! Average reward: {avg_reward:.2f}")
                break
    
    return episode_rewards

# Train
print("Training REINFORCE agent...\n")
reinforce_rewards = train_reinforce(agent, env, num_episodes=500)

Let's visualize the learning progress:

In [ ]:
def plot_rewards(rewards, window=10, title="Training Progress"):
    """Plot episode rewards with moving average."""
    fig, ax = plt.subplots(figsize=(12, 5))
    
    # Plot raw rewards
    ax.plot(rewards, alpha=0.3, color='blue', label='Episode Reward')
    
    # Plot moving average
    moving_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(rewards)), moving_avg, color='blue', linewidth=2, label=f'{window}-Episode Average')
    
    # Add success threshold
    ax.axhline(y=475, color='green', linestyle='--', alpha=0.7, label='Success Threshold (475)')
    
    ax.set_xlabel('Episode', fontsize=12)
    ax.set_ylabel('Episode Reward', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_rewards(reinforce_rewards, window=10, title="REINFORCE Training Progress")

### Key Observations

Notice how the rewards are **noisy** and **high variance**. This is a fundamental challenge with REINFORCE:

- Episodes can have very different returns due to randomness
- Some episodes get lucky with good actions
- The gradient estimates are noisy

**Solution**: Use baselines to reduce variance!

## 6. Baselines - Reducing Variance

A **baseline** is a value we subtract from returns to reduce gradient variance without adding bias.

### The Math

Instead of:
$$\nabla_\theta J = \mathbb{E}[\sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t]$$

We use:
$$\nabla_\theta J = \mathbb{E}[\sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot (G_t - b(s_t))]$$

where $b(s_t)$ is the baseline (commonly the **value function** $V(s_t)$).

### Why It Works

**Intuition**: 
- $G_t - b(s_t)$ is the **advantage** - how much better this return is than expected
- If $G_t > b(s_t)$: action was better than average → increase probability
- If $G_t < b(s_t)$: action was worse than average → decrease probability

**Why it's unbiased**: $\mathbb{E}[\nabla_\theta \log \pi_\theta(a_t|s_t) \cdot b(s_t)] = 0$ (the baseline doesn't depend on actions)

### Simple Baseline

The simplest baseline is the **moving average** of episode returns.

In [ ]:
class REINFORCEWithBaseline:
    """REINFORCE with a simple moving average baseline."""
    
    def __init__(self, policy, lr=0.001, gamma=0.99):
        self.policy = policy
        self.optimizer = optim.Adam(policy.parameters(), lr=lr)
        self.gamma = gamma
        
        # Storage
        self.log_probs = []
        self.rewards = []
        
        # Moving average baseline
        self.baseline = 0
        self.baseline_alpha = 0.1  # Smoothing factor
    
    def select_action(self, state):
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        action, log_prob = self.policy.get_action(state)
        self.log_probs.append(log_prob)
        return action
    
    def store_reward(self, reward):
        self.rewards.append(reward)
    
    def update(self):
        # Compute returns
        returns = compute_returns(self.rewards, self.gamma)
        returns = torch.FloatTensor(returns).to(device)
        
        # Update baseline (moving average of episode returns)
        episode_return = sum(self.rewards)
        self.baseline = (1 - self.baseline_alpha) * self.baseline + self.baseline_alpha * episode_return
        
        # Compute advantages (returns - baseline)
        advantages = returns - self.baseline
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        # Compute policy loss
        policy_loss = []
        for log_prob, advantage in zip(self.log_probs, advantages):
            policy_loss.append(-log_prob * advantage)
        
        policy_loss = torch.stack(policy_loss).sum()
        
        # Update
        self.optimizer.zero_grad()
        policy_loss.backward()
        self.optimizer.step()
        
        # Clear storage
        self.log_probs = []
        self.rewards = []
        
        return policy_loss.item()

# Train with baseline
policy_net_baseline = PolicyNetwork(state_dim, action_dim).to(device)
agent_baseline = REINFORCEWithBaseline(policy_net_baseline, lr=0.001, gamma=0.99)

print("Training REINFORCE with baseline...\n")
baseline_rewards = train_reinforce(agent_baseline, env, num_episodes=500)

Compare vanilla REINFORCE vs REINFORCE with baseline:

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

window = 10

# Vanilla REINFORCE
moving_avg1 = np.convolve(reinforce_rewards, np.ones(window)/window, mode='valid')
ax.plot(range(window-1, len(reinforce_rewards)), moving_avg1, color='blue', linewidth=2, label='REINFORCE (vanilla)')

# REINFORCE with baseline
moving_avg2 = np.convolve(baseline_rewards, np.ones(window)/window, mode='valid')
ax.plot(range(window-1, len(baseline_rewards)), moving_avg2, color='orange', linewidth=2, label='REINFORCE + Baseline')

ax.axhline(y=475, color='green', linestyle='--', alpha=0.7, label='Success Threshold')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Average Reward', fontsize=12)
ax.set_title('REINFORCE: Vanilla vs Baseline', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nBaseline reduces variance and often leads to more stable learning!")

## 7. Actor-Critic - Combining Policy and Value

**Actor-Critic** methods combine the best of both worlds:
- **Actor** (policy): $\pi_\theta(a|s)$ - decides what to do
- **Critic** (value function): $V_\phi(s)$ - evaluates how good states are

### The Advantage Function

The critic learns $V(s)$ and we use it as a baseline:

$$A(s_t, a_t) = r_t + \gamma V(s_{t+1}) - V(s_t)$$

This is called the **TD advantage** or **temporal difference error**.

### Benefits

1. **Lower variance**: Critic provides better baseline than moving average
2. **Online learning**: Can update after each step (not just after episode)
3. **Better for long episodes**: Don't need full episode to update

### Algorithm

1. Sample $(s, a, r, s')$ from environment
2. Compute advantage: $A = r + \gamma V(s') - V(s)$
3. Update actor: $\theta \leftarrow \theta + \alpha_\theta A \nabla_\theta \log \pi_\theta(a|s)$
4. Update critic: $\phi \leftarrow \phi - \alpha_\phi \nabla_\phi (V_\phi(s) - (r + \gamma V(s')))^2$

In [ ]:
class ValueNetwork(nn.Module):
    """Value network (critic) that estimates V(s)."""
    
    def __init__(self, state_dim, hidden_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)  # Output single value
    
    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        value = self.fc3(x)
        return value

# Test value network
value_net = ValueNetwork(state_dim).to(device)
state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)

with torch.no_grad():
    value = value_net(state_tensor)

print(f"State value estimate: {value.item():.4f}")
print(f"(Untrained, so it's random)")

Evaluate the model on the test set.

In [ ]:
class ActorCritic:
    """Actor-Critic algorithm."""
    
    def __init__(self, actor, critic, actor_lr=0.001, critic_lr=0.005, gamma=0.99):
        self.actor = actor
        self.critic = critic
        self.actor_optimizer = optim.Adam(actor.parameters(), lr=actor_lr)
        self.critic_optimizer = optim.Adam(critic.parameters(), lr=critic_lr)
        self.gamma = gamma
    
    def select_action(self, state):
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        action, log_prob = self.actor.get_action(state)
        return action, log_prob, state
    
    def update(self, state, log_prob, reward, next_state, done):
        """Update actor and critic after each step."""
        next_state = torch.FloatTensor(next_state).unsqueeze(0).to(device)
        
        # Compute TD error (advantage)
        with torch.no_grad():
            value = self.critic(state)
            next_value = torch.tensor([[0.0]], device=device) if done else self.critic(next_state)
            td_target = reward + self.gamma * next_value
            advantage = td_target - value
        
        # Update actor (policy)
        actor_loss = -log_prob * advantage
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()
        
        # Update critic (value function)
        value = self.critic(state)
        critic_loss = F.mse_loss(value, td_target)
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()
        
        return actor_loss.item(), critic_loss.item()

# Create Actor-Critic agent
actor = PolicyNetwork(state_dim, action_dim).to(device)
critic = ValueNetwork(state_dim).to(device)
ac_agent = ActorCritic(actor, critic, actor_lr=0.001, critic_lr=0.005)
print("Actor-Critic agent created!")

Display the output.

In [ ]:
def train_actor_critic(agent, env, num_episodes=500, print_every=50):
    """Train Actor-Critic agent."""
    episode_rewards = []
    running_reward = deque(maxlen=100)
    
    for episode in range(num_episodes):
        state, _ = env.reset()
        episode_reward = 0
        done = False
        
        while not done:
            # Select action
            action, log_prob, state_tensor = agent.select_action(state)
            
            # Take action
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Update agent (online update after each step!)
            agent.update(state_tensor, log_prob, reward, next_state, done)
            
            state = next_state
            episode_reward += reward
        
        # Track progress
        episode_rewards.append(episode_reward)
        running_reward.append(episode_reward)
        
        if (episode + 1) % print_every == 0:
            avg_reward = np.mean(running_reward)
            print(f"Episode {episode+1}/{num_episodes} | Avg Reward: {avg_reward:.2f}")
            
            if avg_reward > 475:
                print(f"\n✓ Solved in {episode+1} episodes! Average reward: {avg_reward:.2f}")
                break
    
    return episode_rewards

# Train Actor-Critic
print("Training Actor-Critic...\n")
ac_rewards = train_actor_critic(ac_agent, env, num_episodes=500)

Compare all three methods:

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

window = 10

# Plot all methods
methods = [
    (reinforce_rewards, 'REINFORCE', 'blue'),
    (baseline_rewards, 'REINFORCE + Baseline', 'orange'),
    (ac_rewards, 'Actor-Critic', 'green')
]

for rewards, label, color in methods:
    moving_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(rewards)), moving_avg, linewidth=2, label=label, color=color)

ax.axhline(y=475, color='red', linestyle='--', alpha=0.7, label='Success Threshold')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Average Reward', fontsize=12)
ax.set_title('Policy Gradient Methods Comparison', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nActor-Critic often learns faster due to online updates and learned baseline!")

## 8. PPO Basics - Modern Policy Optimization

**Proximal Policy Optimization (PPO)** is a state-of-the-art policy gradient method that solves a key problem:

**Problem**: Policy gradients can take steps that are **too large**, causing the policy to change drastically and hurt performance.

**Solution**: PPO uses a **clipped objective** to prevent large policy updates.

### The PPO Objective

Instead of directly using advantages, PPO uses:

$$L^{\text{CLIP}}(\theta) = \mathbb{E}\left[\min\left(r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t\right)\right]$$

where:
- $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$ is the **probability ratio**
- $\epsilon$ is a small constant (typically 0.2)

### Intuition

- If advantage $A_t > 0$ (good action): increase $\pi_\theta(a|s)$, but **not too much** (clip at $1+\epsilon$)
- If advantage $A_t < 0$ (bad action): decrease $\pi_\theta(a|s)$, but **not too much** (clip at $1-\epsilon$)

This **trust region** approach keeps policy updates conservative and stable.

### Key Features

1. **Multiple epochs**: Reuse collected data for several updates
2. **Minibatch updates**: Sample minibatches from episode buffer
3. **Clipped objective**: Prevent destructive updates
4. **Generalized Advantage Estimation (GAE)**: Better advantage estimates

In [ ]:
class SimplePPO:
    """Simplified PPO implementation."""
    
    def __init__(self, actor, critic, actor_lr=0.0003, critic_lr=0.001, gamma=0.99, 
                 eps_clip=0.2, K_epochs=4):
        self.actor = actor
        self.critic = critic
        self.actor_optimizer = optim.Adam(actor.parameters(), lr=actor_lr)
        self.critic_optimizer = optim.Adam(critic.parameters(), lr=critic_lr)
        
        self.gamma = gamma
        self.eps_clip = eps_clip
        self.K_epochs = K_epochs
        
        # Episode buffer
        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.dones = []
    
    def select_action(self, state):
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        action, log_prob = self.actor.get_action(state_tensor)
        
        # Store for training
        self.states.append(state)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        
        return action
    
    def store_transition(self, reward, done):
        self.rewards.append(reward)
        self.dones.append(done)
    
    def update(self):
        """Update using collected episode data."""
        # Convert to tensors
        states = torch.FloatTensor(np.array(self.states)).to(device)
        actions = torch.LongTensor(self.actions).to(device)
        old_log_probs = torch.stack(self.log_probs).detach()
        
        # Compute returns and advantages
        returns = compute_returns(self.rewards, self.gamma)
        returns = torch.FloatTensor(returns).to(device)
        
        # Normalize returns
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        
        # Multiple epochs of updates
        for _ in range(self.K_epochs):
            # Evaluate actions with current policy
            logits = self.actor(states)
            dist = Categorical(logits=logits)
            new_log_probs = dist.log_prob(actions)
            
            # Compute ratio (pi_theta / pi_theta_old)
            ratios = torch.exp(new_log_probs - old_log_probs)
            
            # Compute advantages (using returns as simple approximation)
            values = self.critic(states).squeeze()
            advantages = returns - values.detach()
            
            # PPO clipped objective
            surr1 = ratios * advantages
            surr2 = torch.clamp(ratios, 1 - self.eps_clip, 1 + self.eps_clip) * advantages
            actor_loss = -torch.min(surr1, surr2).mean()
            
            # Value loss
            critic_loss = F.mse_loss(values, returns)
            
            # Update actor
            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            self.actor_optimizer.step()
            
            # Update critic
            self.critic_optimizer.zero_grad()
            critic_loss.backward()
            self.critic_optimizer.step()
        
        # Clear buffer
        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.dones = []

# Create PPO agent
ppo_actor = PolicyNetwork(state_dim, action_dim).to(device)
ppo_critic = ValueNetwork(state_dim).to(device)
ppo_agent = SimplePPO(ppo_actor, ppo_critic, eps_clip=0.2, K_epochs=4)
print("PPO agent created!")

Display the output.

In [ ]:
def train_ppo(agent, env, num_episodes=500, print_every=50):
    """Train PPO agent."""
    episode_rewards = []
    running_reward = deque(maxlen=100)
    
    for episode in range(num_episodes):
        state, _ = env.reset()
        episode_reward = 0
        done = False
        
        # Collect episode
        while not done:
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            agent.store_transition(reward, done)
            
            state = next_state
            episode_reward += reward
        
        # Update after episode
        agent.update()
        
        # Track progress
        episode_rewards.append(episode_reward)
        running_reward.append(episode_reward)
        
        if (episode + 1) % print_every == 0:
            avg_reward = np.mean(running_reward)
            print(f"Episode {episode+1}/{num_episodes} | Avg Reward: {avg_reward:.2f}")
            
            if avg_reward > 475:
                print(f"\n✓ Solved in {episode+1} episodes! Average reward: {avg_reward:.2f}")
                break
    
    return episode_rewards

# Train PPO
print("Training PPO...\n")
ppo_rewards = train_ppo(ppo_agent, env, num_episodes=500)

## 9. Final Comparison

Let's compare all four policy gradient methods we've implemented:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

window = 10

methods = [
    (reinforce_rewards, 'REINFORCE', 'blue'),
    (baseline_rewards, 'REINFORCE + Baseline', 'orange'),
    (ac_rewards, 'Actor-Critic', 'green'),
    (ppo_rewards, 'PPO', 'red')
]

# Plot 1: Learning curves
for rewards, label, color in methods:
    moving_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax1.plot(range(window-1, len(rewards)), moving_avg, linewidth=2, label=label, color=color)

ax1.axhline(y=475, color='black', linestyle='--', alpha=0.5, label='Success Threshold')
ax1.set_xlabel('Episode', fontsize=12)
ax1.set_ylabel('Average Reward', fontsize=12)
ax1.set_title('Learning Curves Comparison', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Episodes to solve
def episodes_to_solve(rewards, threshold=475, window=100):
    """Find first episode where moving average exceeds threshold."""
    if len(rewards) < window:
        return len(rewards)
    for i in range(window, len(rewards)):
        if np.mean(rewards[i-window:i]) > threshold:
            return i
    return len(rewards)

solve_episodes = [episodes_to_solve(r) for r, _, _ in methods]
labels = [label for _, label, _ in methods]
colors = [color for _, _, color in methods]

bars = ax2.bar(labels, solve_episodes, color=colors, alpha=0.7)
ax2.set_ylabel('Episodes to Solve', fontsize=12)
ax2.set_title('Sample Efficiency Comparison', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for bar, episodes in zip(bars, solve_episodes):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(episodes)}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nSummary:")
for (_, label, _), episodes in zip(methods, solve_episodes):
    print(f"{label:25} → solved in {episodes:3} episodes")

## 10. Visualizing Learned Policies

Let's visualize what the trained policies have learned by examining their action probabilities across different states:

In [ ]:
def visualize_policy(policy, env, num_samples=50):
    """Visualize policy's action probabilities across state space."""
    angles = []
    angular_velocities = []
    action_probs_left = []
    
    # Sample states
    for _ in range(num_samples):
        state, _ = env.reset()
        
        # Run for a few steps to get diverse states
        for _ in range(np.random.randint(1, 50)):
            action = env.action_space.sample()
            state, _, terminated, truncated, _ = env.step(action)
            if terminated or truncated:
                break
        
        # Get policy probabilities for this state
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = policy(state_tensor)
            probs = F.softmax(logits, dim=-1)
        
        angles.append(state[2])  # Pole angle
        angular_velocities.append(state[3])  # Pole angular velocity
        action_probs_left.append(probs[0, 0].item())  # Prob of moving left
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    scatter = ax.scatter(angles, angular_velocities, c=action_probs_left, 
                        s=100, cmap='RdYlBu', vmin=0, vmax=1, alpha=0.6, edgecolors='black')
    
    ax.set_xlabel('Pole Angle', fontsize=12)
    ax.set_ylabel('Pole Angular Velocity', fontsize=12)
    ax.set_title('Policy Visualization: Probability of Moving Left', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.axvline(x=0, color='black', linestyle='--', alpha=0.3)
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
    
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('P(move left)', fontsize=11)
    
    plt.tight_layout()
    plt.show()

print("Visualizing PPO policy:\n")
visualize_policy(ppo_agent.actor, env, num_samples=100)

print("\nInterpretation:")
print("- Blue regions: Policy prefers moving LEFT")
print("- Red regions: Policy prefers moving RIGHT")
print("- The policy learned to counteract the pole's motion!")

## 11. Key Takeaways

### Core Concepts

✓ **Policy Gradients** directly optimize the policy using gradient ascent on expected rewards

✓ **Policy Gradient Theorem** provides the mathematical foundation:
  - Gradient points toward actions with higher rewards
  - Estimated using Monte Carlo sampling

✓ **REINFORCE** is the simplest algorithm:
  - Collect full episodes
  - Update using returns
  - High variance but unbiased

✓ **Baselines** reduce variance without adding bias:
  - Subtract expected return from actual return
  - Advantage = how much better than expected
  - Leads to more stable learning

✓ **Actor-Critic** combines policy and value:
  - Actor (policy) decides actions
  - Critic (value function) evaluates states
  - Lower variance, online updates

✓ **PPO** adds stability through clipping:
  - Prevents destructive policy updates
  - Multiple epochs per episode
  - State-of-the-art performance

### Comparison Summary

| Method | Variance | Sample Efficiency | Stability | Complexity |
|--------|----------|-------------------|-----------|------------|
| REINFORCE | High | Low | Low | Simple |
| REINFORCE + Baseline | Medium | Medium | Medium | Simple |
| Actor-Critic | Medium | Medium-High | Medium | Moderate |
| PPO | Low | High | High | Complex |

### When to Use Each

- **REINFORCE**: Simple problems, educational purposes
- **REINFORCE + Baseline**: When you need stability without much complexity
- **Actor-Critic**: Good balance of performance and complexity
- **PPO**: Production systems, difficult RL tasks, continuous control

### Advantages of Policy Gradients

1. **Continuous actions**: Natural handling of continuous action spaces
2. **Stochastic policies**: Can learn probabilistic strategies
3. **Convergence**: Guaranteed to converge to local optimum (with proper learning rates)
4. **No Q-function**: Direct policy learning can be simpler

### Challenges

1. **High variance**: Gradient estimates are noisy
2. **Sample inefficiency**: Needs many environment interactions
3. **Local optima**: Can get stuck in suboptimal policies
4. **Sensitive to hyperparameters**: Learning rate, baseline choice matter

## 12. Extensions & Next Steps

### Advanced Topics

1. **Generalized Advantage Estimation (GAE)**: Better advantage computation with λ-returns
2. **Trust Region Policy Optimization (TRPO)**: PPO's predecessor with theoretical guarantees
3. **Soft Actor-Critic (SAC)**: Entropy-regularized off-policy method
4. **Continuous action spaces**: Policy gradient with Gaussian policies
5. **Multi-agent RL**: Policy gradients in competitive/cooperative settings

### Experiments to Try

1. **Different environments**: Try other Gym environments (Pendulum, LunarLander)
2. **Hyperparameter tuning**: Experiment with learning rates, hidden sizes, γ
3. **Network architecture**: Try different policy network architectures
4. **Entropy regularization**: Add entropy bonus to encourage exploration
5. **Multiple workers**: Parallel environment collection (A3C, PPO)

### Resources

- **Spinning Up in Deep RL** (OpenAI): Excellent educational resource
- **Sutton & Barto**: "Reinforcement Learning: An Introduction" (Chapter 13)
- **PPO Paper**: "Proximal Policy Optimization Algorithms" (Schulman et al., 2017)
- **A3C Paper**: "Asynchronous Methods for Deep RL" (Mnih et al., 2016)

## Summary

You've learned the complete policy gradient story:

1. ✓ **Policy Gradient Theorem** - Why directly optimizing policies works
2. ✓ **REINFORCE** - Monte Carlo policy gradient
3. ✓ **Baselines** - Variance reduction through advantages
4. ✓ **Actor-Critic** - Combining policy and value learning
5. ✓ **PPO** - Modern stable policy optimization

These methods form the foundation of modern RL and are used in:
- Robot control (continuous actions)
- Game playing (stochastic strategies)
- Dialogue systems (natural language generation)
- Autonomous driving (complex decision making)

**Key insight**: Policy gradients learn by trying actions, measuring outcomes, and adjusting probabilities - a natural learning paradigm that mirrors how humans and animals learn through trial and error!